In [3]:
"""Paper-faithful SAOCP (Algorithms 1-2) adapted to four energy datasets.

Implementation of Bhatnagar, Wang, Xiong & Bai (ICML 2023), Algorithm 1
(SAOCP) with Algorithm 2 (SF-OGD) and the implementation choices used by
https://github.com/salesforce/online_conformal/blob/main/online_conformal/saocp.py

Differences from the original M4 experiment (not changes to SAOCP):
  - Base point forecast = midpoint of two rolling LSTM quantile forecasts.
  - True radius S_t = absolute residual |y_t - point_prediction_t|.
  - Prediction interval = [point_prediction_t - radius_t, point_prediction_t + radius_t].
  - H=1 sequential forecasting (the paper uses multiple independent horizons).
  - Optional online LSTM fine-tuning every TRAIN_EVERY steps.
  - Calibration scale = sqrt(3) * max initial absolute residual (official code).
  - Initial calibration scores warm-start SAOCP, without being evaluated.
  - StandardScaler is fit on the prefix only; no future leakage.
  - The published coverage theorem for SAOCP uses a RANDOMIZED expert variant;
    this script uses the deterministic aggregate of the official implementation.

Run from your original experiment directory, with Datasets and Helper_Scripts
in their original relative locations:
    python run_saocp_energy.py
    # Edit DEFAULT_DATASETS and configuration constants below to customize.

The actual datasets and Helper_Scripts are not embedded in the uploaded .ipynb.
Requires numpy, pandas, scikit-learn, torch, and your models_TLMR.py.
The original Helper_Scripts.calculate_metrics.py is REQUIRED for exact metric parity.
"""
from __future__ import annotations

import math
import random
import time
import sys
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd


SEED = 42
SEQ_LEN = 48
EPOCHS = 200
LR = 0.001
BATCH_SIZE = 32
ALPHA = 0.10
SPLIT = 0.50
LIFETIME = 8             # g in paper Eq. (8), authors' official default
TRAIN_EVERY = 6           # preserve notebook update cadence
DOWNSAMPLE_STRIDE = 4     # grid-only y[::4] (same as pasted four-dataset code)
SAVE_TRAJECTORIES = True
# All settings are configured here. No command-line arguments are required.
TINIT_OVERRIDE = 5000       # same fixed TINIT as the four-dataset experiment
SCALER_MODE = "prefix"     # fit only on observations available before online evaluation
DEVICE = None             # None selects CUDA automatically when available
LOG_EVERY = 200

# The local window and CWC eta are imported from the SAME helper as ACI/LACI.
# Do not define alternate local-metric formulas here.
TRAJECTORY_DIR = "./trajectories/LSTM_SAOCP_paper"
OUTPUT_CSV = "./results_SAOCP_paper.csv"
RUN_SELF_TEST = False      # True runs internal algorithm checks instead of experiments

# DEFAULT_DATASETS = [
#     "../Datasets/50Hertz.csv",
#     "../Datasets/Amprion.csv",
#     "../Datasets/TenneTTSO.csv",
#     "../Datasets/TransnetBW.csv",
# ]

DEFAULT_DATASETS = [
    "../Datasets/50Hertz.csv",
    "../Datasets/Amprion.csv",
    "../Datasets/TenneTTSO.csv",
    "../Datasets/TransnetBW.csv",
]



GRID_NAMES = {"50Hertz", "Amprion", "TenneTTSO", "TransnetBW"}


def pinball(y: float, s: float, coverage: float) -> float:
    """Paper (1): (1-alpha)-pinball loss with y=true score, s=radius."""
    return max(coverage * (y - s), (1.0 - coverage) * (s - y))


def pinball_grad(y: float, s: float, coverage: float) -> float:
    """Authors' tie convention: zero subgradient when y == s."""
    return (-coverage if y > s else (1.0 - coverage if y < s else 0.0))


def two_adic_valuation(n: int) -> int:
    assert n >= 1
    return (n & -n).bit_length() - 1


@dataclass
class SFOGDExpert:
    """Individual SF-OGD expert; radius is in normalized target units."""
    start: int
    scale: float
    alpha: float
    radius: float
    g: int
    grad_norm: float = 0.0
    z: float = 0.0
    wz: float = 0.0
    age: int = 0

    @property
    def lifetime(self) -> int:
        return self.g * (2 ** two_adic_valuation(self.start))

    @property
    def expired(self) -> bool:
        # Matches author's code: expired when age > lifetime.
        return self.age > self.lifetime

    @property
    def wealth_weight(self) -> float:
        return 0.0 if self.age == 0 else (self.z / self.age) * (1.0 + self.wz)

    def update(self, score: float, meta_loss: float) -> None:
        old_weight = self.wealth_weight
        loss_expert = pinball(score, self.radius, 1.0 - self.alpha)
        # Matches official _OGD.update, including its normalization and clipping.
        gain = (meta_loss - loss_expert) / self.scale / max(self.alpha, 1.0 - self.alpha)
        gain = float(np.clip(gain, -float(old_weight > 0.0), 1.0))
        self.z += gain
        self.wz += gain * old_weight
        self.age += 1
        grad = pinball_grad(score, self.radius, 1.0 - self.alpha)
        self.grad_norm += grad * grad
        if self.grad_norm != 0.0:
            eta = self.scale / math.sqrt(3.0)
            self.radius = max(0.0, self.radius - eta * grad / math.sqrt(self.grad_norm))


class SAOCP:
    """One-step deterministic SAOCP following salesforce/online_conformal/saocp.py.

    Call predict() before seeing the current y. Then call update(score).
    update() creates an expert at the current time, aggregates meta loss,
    updates active experts and increments time; it never sees future scores.
    """
    def __init__(self, alpha: float, scale: float, lifetime: int = LIFETIME):
        if not 0.0 < alpha < 1.0:
            raise ValueError("alpha must lie strictly between zero and one")
        if not np.isfinite(scale) or scale <= 0.0:
            raise ValueError("scale must be finite and positive")
        if lifetime < 1:
            raise ValueError("lifetime must be positive")
        self.alpha = float(alpha)
        self.scale = float(scale)
        self.lifetime = int(lifetime)
        self.t = 1
        self.experts: dict[int, SFOGDExpert] = {}

    def probabilities(self) -> dict[int, float]:
        if not self.experts:
            return {}
        prior = {
            k: 1.0 / (k * k * (1.0 + math.floor(math.log2(k))))
            for k in self.experts
        }
        total_prior = sum(prior.values())
        prior = {k: v / total_prior for k, v in prior.items()}
        unnorm = {k: prior[k] * max(0.0, e.wealth_weight)
                  for k, e in self.experts.items()}
        total = sum(unnorm.values())
        return ({k: v / total for k, v in unnorm.items()}
                if total > 0.0 else prior)

    def predict(self) -> float:
        p = self.probabilities()
        return float(sum(p[k] * self.experts[k].radius for k in p))

    def update(self, score: float) -> None:
        if not np.isfinite(score) or score < 0.0:
            raise ValueError(f"Expected a nonnegative finite conformity score: {score}")
        # Official code initializes the new expert from the CURRENT aggregate,
        # then aggregates once more INCLUDING the newly created expert.
        init_radius = self.predict()
        for k in [k for k, v in self.experts.items() if v.expired]:
            del self.experts[k]
        self.experts[self.t] = SFOGDExpert(
            start=self.t, scale=self.scale, alpha=self.alpha,
            radius=init_radius, g=self.lifetime)
        meta_radius = self.predict()
        meta_loss = pinball(float(score), meta_radius, 1.0 - self.alpha)
        for expert in self.experts.values():
            expert.update(float(score), meta_loss)
        self.t += 1


def point_forecast(lower, upper):
    """Use the midpoint of sorted LSTM quantiles as an H=1 point prediction."""
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)
    return 0.5 * (lower + upper)


def nonconformity(y: np.ndarray, lower: np.ndarray, upper: np.ndarray) -> np.ndarray:
    """Paper Eq. (13), H=1: true radius = absolute point-forecast residual."""
    return np.abs(np.asarray(y, dtype=float) - point_forecast(lower, upper))


def read_series(filepath: str) -> np.ndarray:
    path = Path(filepath)
    if not path.is_file():
        raise FileNotFoundError(f"Missing dataset: {path}")
    df = pd.read_csv(path)
    name = path.stem
    if name not in GRID_NAMES:
        raise ValueError(f"Only the four grid datasets are supported: {path}")
    # Preserve previous experiment preprocessing: flatten columns after first, downsample.
    # Verify actual column chronology: flattening multiple variables may be inappropriate.
    y = df.iloc[:, 1:].to_numpy().flatten().astype(float)[::DOWNSAMPLE_STRIDE]
    if not len(y) or not np.all(np.isfinite(y)):
        raise ValueError(f"Nonfinite or empty series: {path}")
    return y


def build_sequences(series: np.ndarray, seq_len: int) -> tuple[np.ndarray, np.ndarray]:
    # A sliding view avoids the very slow Python list construction in notebook.
    if len(series) <= seq_len:
        raise ValueError("Series too short for SEQ_LEN")
    windows = np.lib.stride_tricks.sliding_window_view(series, seq_len + 1)
    X = np.ascontiguousarray(windows[:, :-1], dtype=np.float32)
    y = np.asarray(windows[:, -1], dtype=np.float32)
    return X, y


def seed_everything(seed: int) -> None:
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def setup_models(device: str):
    import torch
    if str(Path.cwd()) not in sys.path:
        sys.path.insert(0, str(Path.cwd()))
    try:
        from Helper_Scripts.models_TLMR import LSTMQuantileReg
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Missing Helper_Scripts.models_TLMR.LSTMQuantileReg; run alongside "
            "your existing experiment or adjust import in setup_models()."
        ) from exc
    models = [LSTMQuantileReg().to(device) for _ in range(2)]
    opts = [torch.optim.Adam(m.parameters(), lr=LR) for m in models]
    return models, opts


def fit_models(models, opts, X: np.ndarray, y: np.ndarray, epochs: int,
               device: str, alpha: float) -> None:
    import torch
    from torch.utils.data import DataLoader, TensorDataset
    if epochs < 1:
        return
    data = TensorDataset(torch.as_tensor(X, dtype=torch.float32),
                         torch.as_tensor(y.reshape(-1, 1), dtype=torch.float32))
    loader = DataLoader(data, batch_size=min(BATCH_SIZE, len(data)), shuffle=True)
    for model, opt, q in zip(models, opts, [alpha / 2.0, 1.0 - alpha / 2.0]):
        model.train()
        for _ in range(epochs):
            for bx, by in loader:
                bx, by = bx.to(device), by.to(device)
                pred = model(bx)
                diff = by - pred
                loss = torch.maximum(q * diff, (q - 1.0) * diff).mean()
                opt.zero_grad()
                loss.backward()
                opt.step()


def predict_quantiles(models, X: np.ndarray, device: str,
                      batch_size: int = 4096) -> tuple[np.ndarray, np.ndarray]:
    import torch
    outputs = []
    for model in models:
        model.eval()
        chunks = []
        with torch.no_grad():
            for idx in range(0, len(X), batch_size):
                bx = torch.as_tensor(X[idx:idx + batch_size], dtype=torch.float32,
                                     device=device)
                chunks.append(model(bx).detach().cpu().numpy().reshape(-1))
        outputs.append(np.concatenate(chunks).astype(float))
    # Crossing correction to guarantee a valid nested interval family.
    return np.minimum(outputs[0], outputs[1]), np.maximum(outputs[0], outputs[1])


def rolling_local_trajectories(traj: pd.DataFrame, window: int, alpha: float) -> None:
    """Store per-time local diagnostics without changing the metric definitions.

    Only complete, trailing windows are reported (the first window-1 rows
    remain NaN), matching the usual local-coverage evaluation convention.
    Summary metrics are computed exclusively by the original shared helper.
    """
    if window < 1:
        raise ValueError("WINDOW from calculate_metrics must be positive")
    covered = ((traj["y_l_pred"] <= traj["y_true"]) &
               (traj["y_true"] <= traj["y_u_pred"])).astype(float)
    width = traj["y_u_pred"] - traj["y_l_pred"]
    lp = covered.rolling(window=window, min_periods=window).mean()
    traj["LPICP"] = lp
    traj["LE"] = ((1.0 - alpha) - lp).clip(lower=0.0)
    traj["local_overcoverage"] = (lp - (1.0 - alpha)).clip(lower=0.0)
    traj["local_abs_coverage_error"] = (lp - (1.0 - alpha)).abs()
    traj["local_MPIW"] = width.rolling(window=window, min_periods=window).mean()


def run_dataset(filepath: str) -> dict:
    from sklearn.preprocessing import StandardScaler
    from Helper_Scripts.calculate_metrics import (
        summarize_all_metrics, WINDOW as LOCAL_WINDOW, CWC_ETA
    )
    import torch
    name = Path(filepath).stem
    start_time = time.monotonic()
    seed_everything(SEED)
    raw = read_series(filepath)
    tinit = TINIT_OVERRIDE
    if len(raw) <= SEQ_LEN + tinit:
        raise ValueError(f"{name}: {len(raw)} points cannot support tinit={tinit}, seq_len={SEQ_LEN}")

    # Prefix mode prevents leakage from future observations into normalization.
    scaler = StandardScaler()
    scaler.fit(raw[:SEQ_LEN + tinit].reshape(-1, 1))
    series = scaler.transform(raw.reshape(-1, 1)).ravel()
    X, y = build_sequences(series, SEQ_LEN)
    n_train = int(round(SPLIT * tinit))
    if n_train < 2 or n_train >= tinit:
        raise ValueError("Train/calibration split must leave both nonempty")

    device = DEVICE or ("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{name}: raw={len(raw)} sequences={len(y)} tinit={tinit} "
          f"training={n_train} initial_cal={tinit-n_train} device={device}", flush=True)
    models, opts = setup_models(device)
    fit_models(models, opts, X[:n_train], y[:n_train], EPOCHS, device, ALPHA)

    # Initial calibration is temporal and held out from LSTM fitting.
    cal_lo, cal_hi = predict_quantiles(models, X[n_train:tinit], device)
    cal_scores = nonconformity(y[n_train:tinit], cal_lo, cal_hi)
    max_score = float(np.max(cal_scores))
    # Official scale rule on absolute (not halfwidth-normalized) residuals.
    # Zero-only calibration needs a strictly positive numerical fallback.
    scale = max(max_score * math.sqrt(3), 1e-8)
    saocp = SAOCP(alpha=ALPHA, scale=scale, lifetime=LIFETIME)
    for score in cal_scores:
        saocp.update(float(score))

    rows = []
    for t in range(tinit, len(y)):
        # Keep the existing periodic base-model fine-tuning protocol.
        if t > tinit and (t - tinit) % TRAIN_EVERY == 0:
            # Most recent n_train examples with labels revealed before t.
            fit_models(models, opts, X[t-n_train:t], y[t-n_train:t],
                       1, device, ALPHA)

        lo_q, hi_q = predict_quantiles(models, X[t:t+1], device)
        lo_q, hi_q = float(lo_q[0]), float(hi_q[0])
        radius = saocp.predict()  # BEFORE observing y[t], units: normalized target
        center = float(point_forecast(lo_q, hi_q))
        lower = center - radius
        upper = center + radius
        yt = float(y[t])
        err = int(yt < lower or yt > upper)
        score = abs(yt - center)  # paper true radius, unscaled by quantile width
        rows.append((t - tinit, t, yt, lower, upper, center, lo_q, hi_q,
                     radius, score, err, len(saocp.experts)))
        saocp.update(score)       # AFTER observing y[t]
        if LOG_EVERY and (t - tinit + 1) % LOG_EVERY == 0:
            print(f"  {name}: {t-tinit+1}/{len(y)-tinit} | "
                  f"radius={radius:.5f} | running_coverage="
                  f"{1.0-np.mean([r[10] for r in rows]):.4f}", flush=True)

    columns = ["step", "dataset_index", "y_true", "y_l_pred", "y_u_pred",
               "point_pred", "q_lower", "q_upper", "radius", "score", "err", "active_experts"]
    traj = pd.DataFrame(rows, columns=columns)
    # Invert scale only for target-valued quantities, not counts/indicators.
    for col in ["y_true", "y_l_pred", "y_u_pred", "point_pred", "q_lower", "q_upper"]:
        traj[col] = scaler.inverse_transform(traj[[col]].to_numpy()).reshape(-1)
    traj["radius"] *= float(scaler.scale_[0])
    traj["score"] *= float(scaler.scale_[0])
    traj["alphaTrajectory"] = np.nan  # SAOCP directly learns radius, not alpha_t.
    traj["errSeq"] = traj["err"]
    traj["covered"] = 1 - traj["err"]
    rolling_local_trajectories(traj, LOCAL_WINDOW, ALPHA)

    outdir = Path(TRAJECTORY_DIR)
    outdir.mkdir(parents=True, exist_ok=True)
    if SAVE_TRAJECTORIES:
        traj.to_csv(outdir / f"{name}__SAOCP.csv", index=False)

    truth = traj["y_true"].to_numpy(dtype=float)
    lower = traj["y_l_pred"].to_numpy(dtype=float)
    upper = traj["y_u_pred"].to_numpy(dtype=float)
    elapsed = time.monotonic() - start_time
    # Same evaluator, arguments and metric names as the ACI/LACI experiment.
    metrics = summarize_all_metrics(
        truth, lower, upper,
        alpha=ALPHA, w=LOCAL_WINDOW, eta=CWC_ETA, time_taken=elapsed
    )
    metrics["time_sec"] = float(elapsed)
    metrics["dataset"] = name + ".csv"
    metrics["method"] = "SAOCP"
    metrics["scale_D_from_cal"] = float(max_score * scaler.scale_[0])  # original units
    metrics["g_lifetime"] = LIFETIME
    metrics["mean_radius"] = float(traj["radius"].mean())
    metrics["median_radius"] = float(traj["radius"].median())
    base_covered = ((traj["y_true"] >= traj["q_lower"]) &
                    (traj["y_true"] <= traj["q_upper"]))
    metrics["base_quantile_PICP"] = float(base_covered.mean())
    metric_names = ["PICP", "MPIW", "MLPICP", "LPICE", "LPIA",
                    "LMPIW", "LCWC", "STD_LE"]
    print(f"[{name}/SAOCP] " + " | ".join(
        f"{k}={float(metrics[k]):.4f}" for k in metric_names if k in metrics
    ) + f" | time_sec={elapsed:.2f}", flush=True)
    return metrics


def self_test() -> None:
    # Regression tests of the deterministic official-style SAOCP machinery.
    a = SAOCP(alpha=0.1, scale=math.sqrt(3), lifetime=8)
    scores = np.r_[np.full(70, 0.5), np.full(70, 2.0), np.full(70, 0.3)]
    predicted = []
    for score in scores:
        r = a.predict()
        predicted.append(r)
        assert np.isfinite(r) and r >= 0
        a.update(float(score))
        probs = a.probabilities()
        assert abs(sum(probs.values()) - 1.0) < 1e-9
        assert all(v >= 0.0 for v in probs.values())
    assert len(a.experts) < 100
    assert np.mean(predicted[100:140]) > np.mean(predicted[25:60])
    assert np.allclose(nonconformity(np.array([0., 2., 5.]),
                                      np.array([-1., -1., 3.]),
                                      np.array([1., 1., 7.])), [0., 2., 0.])
    # Explicit interval nesting and inclusion equivalence.
    for y in [4., 8., 10., 14., 16.]:
        s = abs(y - 10.)
        for r in [0., 2., 4., 6.]:
            assert ((10. - r <= y <= 10. + r) == (s <= r))
    # No score width normalization: same point residual gives same score.
    assert nonconformity(np.array(12.), np.array(9.), np.array(11.)) == 2.
    assert nonconformity(np.array(12.), np.array(0.), np.array(20.)) == 2.
    print("PASS: SAOCP expert updates, probabilities, shifts, absolute scores, nested intervals")


def main() -> None:
    if RUN_SELF_TEST:
        self_test()
        return
    if TRAIN_EVERY < 1:
        raise ValueError("TRAIN_EVERY must be >= 1")
    if SCALER_MODE != "prefix":
        raise ValueError("Only leakage-free SCALER_MODE='prefix' is supported")
    if DOWNSAMPLE_STRIDE < 1:
        raise ValueError("DOWNSAMPLE_STRIDE must be >= 1")
    if not DEFAULT_DATASETS:
        raise ValueError("DEFAULT_DATASETS must include at least one file")
    missing = [p for p in DEFAULT_DATASETS if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("Datasets missing (no synthetic fallback):\n" + "\n".join(missing))
    metrics = []
    for path in DEFAULT_DATASETS:
        row = run_dataset(path)
        metrics.append(row)
        pd.DataFrame(metrics).to_csv(OUTPUT_CSV, index=False)
    # Match the column ordering in your ACI/DtACI/AgACI/LACI notebook.
    results_df = pd.DataFrame(metrics)
    metric_order = ["dataset", "method", "PICP", "MPIW", "MLPICP", "LPICE",
                    "LPIA", "LMPIW", "LCWC", "STD_LE", "n_local_windows",
                    "time_sec"]
    ordered = [k for k in metric_order if k in results_df.columns]
    results_df = results_df[ordered + [k for k in results_df if k not in ordered]]
    results_df.to_csv(OUTPUT_CSV, index=False)
    print("\nSaved summary:", OUTPUT_CSV)
    if SAVE_TRAJECTORIES:
        print("Saved trajectories:", TRAJECTORY_DIR)
    print(results_df.to_string(index=False))


if __name__ == "__main__":
    main()



50Hertz: raw=9528 sequences=9480 tinit=5000 training=2500 initial_cal=2500 device=cuda
  50Hertz: 200/4480 | radius=0.11804 | running_coverage=0.9050
  50Hertz: 400/4480 | radius=0.35422 | running_coverage=0.9100
  50Hertz: 600/4480 | radius=0.31760 | running_coverage=0.9183
  50Hertz: 800/4480 | radius=0.35565 | running_coverage=0.9163
  50Hertz: 1000/4480 | radius=0.29671 | running_coverage=0.9190
  50Hertz: 1200/4480 | radius=0.04875 | running_coverage=0.9225
  50Hertz: 1400/4480 | radius=0.14276 | running_coverage=0.9221
  50Hertz: 1600/4480 | radius=0.25052 | running_coverage=0.9194
  50Hertz: 1800/4480 | radius=0.09589 | running_coverage=0.9206
  50Hertz: 2000/4480 | radius=0.10640 | running_coverage=0.9195
  50Hertz: 2200/4480 | radius=0.27784 | running_coverage=0.9209
  50Hertz: 2400/4480 | radius=0.22218 | running_coverage=0.9213
  50Hertz: 2600/4480 | radius=0.37841 | running_coverage=0.9212
  50Hertz: 2800/4480 | radius=0.31022 | running_coverage=0.9229
  50Hertz: 3000/4480

In [4]:

from __future__ import annotations

import math
import random
import time
import sys
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd


SEED = 42
SEQ_LEN = 48
EPOCHS = 200
LR = 0.001
BATCH_SIZE = 32
ALPHA = 0.10
SPLIT = 0.50
LIFETIME = 8             # g in paper Eq. (8), authors' official default
TRAIN_EVERY = 6           # preserve notebook update cadence
SAVE_TRAJECTORIES = True
# All settings are configured here. No command-line arguments are required.
TINIT_OVERRIDE = 200       # same fixed TINIT as hydropower experiment
SCALER_MODE = "prefix"     # fit only on observations available before online evaluation
DEVICE = None             # None selects CUDA automatically when available
LOG_EVERY = 200

# The local window and CWC eta are imported from the SAME helper as ACI/LACI.
# Do not define alternate local-metric formulas here.
TRAJECTORY_DIR = "./trajectories/LSTM_SAOCP_hydropower_paper"
OUTPUT_CSV = "./results_SAOCP_hydropower_paper.csv"
RUN_SELF_TEST = False      # True runs internal algorithm checks instead of experiments

DEFAULT_DATASETS = [
    "../Datasets/hydropower_dataset.csv",
]
HYDRO_NAMES = {"hydropower_dataset"}


def pinball(y: float, s: float, coverage: float) -> float:
    """Paper (1): (1-alpha)-pinball loss with y=true score, s=radius."""
    return max(coverage * (y - s), (1.0 - coverage) * (s - y))


def pinball_grad(y: float, s: float, coverage: float) -> float:
    """Authors' tie convention: zero subgradient when y == s."""
    return (-coverage if y > s else (1.0 - coverage if y < s else 0.0))


def two_adic_valuation(n: int) -> int:
    assert n >= 1
    return (n & -n).bit_length() - 1


@dataclass
class SFOGDExpert:
    """Individual SF-OGD expert; radius is in normalized target units."""
    start: int
    scale: float
    alpha: float
    radius: float
    g: int
    grad_norm: float = 0.0
    z: float = 0.0
    wz: float = 0.0
    age: int = 0

    @property
    def lifetime(self) -> int:
        return self.g * (2 ** two_adic_valuation(self.start))

    @property
    def expired(self) -> bool:
        # Matches author's code: expired when age > lifetime.
        return self.age > self.lifetime

    @property
    def wealth_weight(self) -> float:
        return 0.0 if self.age == 0 else (self.z / self.age) * (1.0 + self.wz)

    def update(self, score: float, meta_loss: float) -> None:
        old_weight = self.wealth_weight
        loss_expert = pinball(score, self.radius, 1.0 - self.alpha)
        # Matches official _OGD.update, including its normalization and clipping.
        gain = (meta_loss - loss_expert) / self.scale / max(self.alpha, 1.0 - self.alpha)
        gain = float(np.clip(gain, -float(old_weight > 0.0), 1.0))
        self.z += gain
        self.wz += gain * old_weight
        self.age += 1
        grad = pinball_grad(score, self.radius, 1.0 - self.alpha)
        self.grad_norm += grad * grad
        if self.grad_norm != 0.0:
            eta = self.scale / math.sqrt(3.0)
            self.radius = max(0.0, self.radius - eta * grad / math.sqrt(self.grad_norm))


class SAOCP:
    """One-step deterministic SAOCP following salesforce/online_conformal/saocp.py.

    Call predict() before seeing the current y. Then call update(score).
    update() creates an expert at the current time, aggregates meta loss,
    updates active experts and increments time; it never sees future scores.
    """
    def __init__(self, alpha: float, scale: float, lifetime: int = LIFETIME):
        if not 0.0 < alpha < 1.0:
            raise ValueError("alpha must lie strictly between zero and one")
        if not np.isfinite(scale) or scale <= 0.0:
            raise ValueError("scale must be finite and positive")
        if lifetime < 1:
            raise ValueError("lifetime must be positive")
        self.alpha = float(alpha)
        self.scale = float(scale)
        self.lifetime = int(lifetime)
        self.t = 1
        self.experts: dict[int, SFOGDExpert] = {}

    def probabilities(self) -> dict[int, float]:
        if not self.experts:
            return {}
        prior = {
            k: 1.0 / (k * k * (1.0 + math.floor(math.log2(k))))
            for k in self.experts
        }
        total_prior = sum(prior.values())
        prior = {k: v / total_prior for k, v in prior.items()}
        unnorm = {k: prior[k] * max(0.0, e.wealth_weight)
                  for k, e in self.experts.items()}
        total = sum(unnorm.values())
        return ({k: v / total for k, v in unnorm.items()}
                if total > 0.0 else prior)

    def predict(self) -> float:
        p = self.probabilities()
        return float(sum(p[k] * self.experts[k].radius for k in p))

    def update(self, score: float) -> None:
        if not np.isfinite(score) or score < 0.0:
            raise ValueError(f"Expected a nonnegative finite conformity score: {score}")
        # Official code initializes the new expert from the CURRENT aggregate,
        # then aggregates once more INCLUDING the newly created expert.
        init_radius = self.predict()
        for k in [k for k, v in self.experts.items() if v.expired]:
            del self.experts[k]
        self.experts[self.t] = SFOGDExpert(
            start=self.t, scale=self.scale, alpha=self.alpha,
            radius=init_radius, g=self.lifetime)
        meta_radius = self.predict()
        meta_loss = pinball(float(score), meta_radius, 1.0 - self.alpha)
        for expert in self.experts.values():
            expert.update(float(score), meta_loss)
        self.t += 1


def point_forecast(lower, upper):
    """Use the midpoint of sorted LSTM quantiles as an H=1 point prediction."""
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)
    return 0.5 * (lower + upper)


def nonconformity(y: np.ndarray, lower: np.ndarray, upper: np.ndarray) -> np.ndarray:
    """Paper Eq. (13), H=1: true radius = absolute point-forecast residual."""
    return np.abs(np.asarray(y, dtype=float) - point_forecast(lower, upper))


def read_series(filepath: str) -> np.ndarray:
    """Hydropower loader matching the supplied experiment.

    Sum actual day/evening peaks, remove missing and zero totals. Filtering
    compresses time: consecutive retained observations may not be adjacent days.
    """
    path = Path(filepath)
    if not path.is_file():
        raise FileNotFoundError(f"Missing dataset: {path}")
    if path.stem not in HYDRO_NAMES:
        raise ValueError(f"Expected hydropower_dataset.csv: {path}")
    df = pd.read_csv(path)
    required = ["Actual Day Peak (MW)", "Actual Eve Peak (MW)"]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"{path}: missing required columns: {missing}")
    day = pd.to_numeric(df[required[0]], errors="coerce")
    evening = pd.to_numeric(df[required[1]], errors="coerce")
    total = day + evening
    y = total.loc[total.notna() & (total != 0)].to_numpy(dtype=float)
    if not len(y) or not np.all(np.isfinite(y)):
        raise ValueError(f"Nonfinite or empty filtered hydropower series: {path}")
    return y


def build_sequences(series: np.ndarray, seq_len: int) -> tuple[np.ndarray, np.ndarray]:
    # A sliding view avoids the very slow Python list construction in notebook.
    if len(series) <= seq_len:
        raise ValueError("Series too short for SEQ_LEN")
    windows = np.lib.stride_tricks.sliding_window_view(series, seq_len + 1)
    X = np.ascontiguousarray(windows[:, :-1], dtype=np.float32)
    y = np.asarray(windows[:, -1], dtype=np.float32)
    return X, y


def seed_everything(seed: int) -> None:
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def setup_models(device: str):
    import torch
    if str(Path.cwd()) not in sys.path:
        sys.path.insert(0, str(Path.cwd()))
    try:
        from Helper_Scripts.models_TLMR import LSTMQuantileReg
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Missing Helper_Scripts.models_TLMR.LSTMQuantileReg; run alongside "
            "your existing experiment or adjust import in setup_models()."
        ) from exc
    models = [LSTMQuantileReg().to(device) for _ in range(2)]
    opts = [torch.optim.Adam(m.parameters(), lr=LR) for m in models]
    return models, opts


def fit_models(models, opts, X: np.ndarray, y: np.ndarray, epochs: int,
               device: str, alpha: float) -> None:
    import torch
    from torch.utils.data import DataLoader, TensorDataset
    if epochs < 1:
        return
    data = TensorDataset(torch.as_tensor(X, dtype=torch.float32),
                         torch.as_tensor(y.reshape(-1, 1), dtype=torch.float32))
    loader = DataLoader(data, batch_size=min(BATCH_SIZE, len(data)), shuffle=True)
    for model, opt, q in zip(models, opts, [alpha / 2.0, 1.0 - alpha / 2.0]):
        model.train()
        for _ in range(epochs):
            for bx, by in loader:
                bx, by = bx.to(device), by.to(device)
                pred = model(bx)
                diff = by - pred
                loss = torch.maximum(q * diff, (q - 1.0) * diff).mean()
                opt.zero_grad()
                loss.backward()
                opt.step()


def predict_quantiles(models, X: np.ndarray, device: str,
                      batch_size: int = 4096) -> tuple[np.ndarray, np.ndarray]:
    import torch
    outputs = []
    for model in models:
        model.eval()
        chunks = []
        with torch.no_grad():
            for idx in range(0, len(X), batch_size):
                bx = torch.as_tensor(X[idx:idx + batch_size], dtype=torch.float32,
                                     device=device)
                chunks.append(model(bx).detach().cpu().numpy().reshape(-1))
        outputs.append(np.concatenate(chunks).astype(float))
    # Crossing correction to guarantee a valid nested interval family.
    return np.minimum(outputs[0], outputs[1]), np.maximum(outputs[0], outputs[1])


def rolling_local_trajectories(traj: pd.DataFrame, window: int, alpha: float) -> None:
    """Store per-time local diagnostics without changing the metric definitions.

    Only complete, trailing windows are reported (the first window-1 rows
    remain NaN), matching the usual local-coverage evaluation convention.
    Summary metrics are computed exclusively by the original shared helper.
    """
    if window < 1:
        raise ValueError("WINDOW from calculate_metrics must be positive")
    covered = ((traj["y_l_pred"] <= traj["y_true"]) &
               (traj["y_true"] <= traj["y_u_pred"])).astype(float)
    width = traj["y_u_pred"] - traj["y_l_pred"]
    lp = covered.rolling(window=window, min_periods=window).mean()
    traj["LPICP"] = lp
    traj["LE"] = ((1.0 - alpha) - lp).clip(lower=0.0)
    traj["local_overcoverage"] = (lp - (1.0 - alpha)).clip(lower=0.0)
    traj["local_abs_coverage_error"] = (lp - (1.0 - alpha)).abs()
    traj["local_MPIW"] = width.rolling(window=window, min_periods=window).mean()


def run_dataset(filepath: str) -> dict:
    from sklearn.preprocessing import StandardScaler
    from Helper_Scripts.calculate_metrics import (
        summarize_all_metrics, WINDOW as LOCAL_WINDOW, CWC_ETA
    )
    import torch
    name = Path(filepath).stem
    start_time = time.monotonic()
    seed_everything(SEED)
    raw = read_series(filepath)
    tinit = TINIT_OVERRIDE
    if len(raw) <= SEQ_LEN + tinit:
        raise ValueError(f"{name}: {len(raw)} points cannot support tinit={tinit}, seq_len={SEQ_LEN}")

    # Prefix mode prevents leakage from future observations into normalization.
    scaler = StandardScaler()
    scaler.fit(raw[:SEQ_LEN + tinit].reshape(-1, 1))
    series = scaler.transform(raw.reshape(-1, 1)).ravel()
    X, y = build_sequences(series, SEQ_LEN)
    n_train = int(round(SPLIT * tinit))
    if n_train < 2 or n_train >= tinit:
        raise ValueError("Train/calibration split must leave both nonempty")

    device = DEVICE or ("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{name}: raw={len(raw)} sequences={len(y)} tinit={tinit} "
          f"training={n_train} initial_cal={tinit-n_train} device={device}", flush=True)
    models, opts = setup_models(device)
    fit_models(models, opts, X[:n_train], y[:n_train], EPOCHS, device, ALPHA)

    # Initial calibration is temporal and held out from LSTM fitting.
    cal_lo, cal_hi = predict_quantiles(models, X[n_train:tinit], device)
    cal_scores = nonconformity(y[n_train:tinit], cal_lo, cal_hi)
    max_score = float(np.max(cal_scores))
    # Official scale rule on absolute (not halfwidth-normalized) residuals.
    # Zero-only calibration needs a strictly positive numerical fallback.
    scale = max(max_score * math.sqrt(3), 1e-8)
    saocp = SAOCP(alpha=ALPHA, scale=scale, lifetime=LIFETIME)
    for score in cal_scores:
        saocp.update(float(score))

    rows = []
    for t in range(tinit, len(y)):
        # Keep the existing periodic base-model fine-tuning protocol.
        if t > tinit and (t - tinit) % TRAIN_EVERY == 0:
            # Most recent n_train examples with labels revealed before t.
            fit_models(models, opts, X[t-n_train:t], y[t-n_train:t],
                       1, device, ALPHA)

        lo_q, hi_q = predict_quantiles(models, X[t:t+1], device)
        lo_q, hi_q = float(lo_q[0]), float(hi_q[0])
        radius = saocp.predict()  # BEFORE observing y[t], units: normalized target
        center = float(point_forecast(lo_q, hi_q))
        lower = center - radius
        upper = center + radius
        yt = float(y[t])
        err = int(yt < lower or yt > upper)
        score = abs(yt - center)  # paper true radius, unscaled by quantile width
        rows.append((t - tinit, t, yt, lower, upper, center, lo_q, hi_q,
                     radius, score, err, len(saocp.experts)))
        saocp.update(score)       # AFTER observing y[t]
        if LOG_EVERY and (t - tinit + 1) % LOG_EVERY == 0:
            print(f"  {name}: {t-tinit+1}/{len(y)-tinit} | "
                  f"radius={radius:.5f} | running_coverage="
                  f"{1.0-np.mean([r[10] for r in rows]):.4f}", flush=True)

    columns = ["step", "dataset_index", "y_true", "y_l_pred", "y_u_pred",
               "point_pred", "q_lower", "q_upper", "radius", "score", "err", "active_experts"]
    traj = pd.DataFrame(rows, columns=columns)
    # Invert scale only for target-valued quantities, not counts/indicators.
    for col in ["y_true", "y_l_pred", "y_u_pred", "point_pred", "q_lower", "q_upper"]:
        traj[col] = scaler.inverse_transform(traj[[col]].to_numpy()).reshape(-1)
    traj["radius"] *= float(scaler.scale_[0])
    traj["score"] *= float(scaler.scale_[0])
    traj["alphaTrajectory"] = np.nan  # SAOCP directly learns radius, not alpha_t.
    traj["errSeq"] = traj["err"]
    traj["covered"] = 1 - traj["err"]
    rolling_local_trajectories(traj, LOCAL_WINDOW, ALPHA)

    outdir = Path(TRAJECTORY_DIR)
    outdir.mkdir(parents=True, exist_ok=True)
    if SAVE_TRAJECTORIES:
        traj.to_csv(outdir / f"{name}__SAOCP.csv", index=False)

    truth = traj["y_true"].to_numpy(dtype=float)
    lower = traj["y_l_pred"].to_numpy(dtype=float)
    upper = traj["y_u_pred"].to_numpy(dtype=float)
    elapsed = time.monotonic() - start_time
    # Same evaluator, arguments and metric names as the ACI/LACI experiment.
    metrics = summarize_all_metrics(
        truth, lower, upper,
        alpha=ALPHA, w=LOCAL_WINDOW, eta=CWC_ETA, time_taken=elapsed
    )
    metrics["time_sec"] = float(elapsed)
    metrics["dataset"] = name + ".csv"
    metrics["method"] = "SAOCP"
    metrics["scale_D_from_cal"] = float(max_score * scaler.scale_[0])  # original units
    metrics["g_lifetime"] = LIFETIME
    metrics["mean_radius"] = float(traj["radius"].mean())
    metrics["median_radius"] = float(traj["radius"].median())
    base_covered = ((traj["y_true"] >= traj["q_lower"]) &
                    (traj["y_true"] <= traj["q_upper"]))
    metrics["base_quantile_PICP"] = float(base_covered.mean())
    metric_names = ["PICP", "MPIW", "MLPICP", "LPICE", "LPIA",
                    "LMPIW", "LCWC", "STD_LE"]
    print(f"[{name}/SAOCP] " + " | ".join(
        f"{k}={float(metrics[k]):.4f}" for k in metric_names if k in metrics
    ) + f" | time_sec={elapsed:.2f}", flush=True)
    return metrics


def self_test() -> None:
    # Regression tests of the deterministic official-style SAOCP machinery.
    a = SAOCP(alpha=0.1, scale=math.sqrt(3), lifetime=8)
    scores = np.r_[np.full(70, 0.5), np.full(70, 2.0), np.full(70, 0.3)]
    predicted = []
    for score in scores:
        r = a.predict()
        predicted.append(r)
        assert np.isfinite(r) and r >= 0
        a.update(float(score))
        probs = a.probabilities()
        assert abs(sum(probs.values()) - 1.0) < 1e-9
        assert all(v >= 0.0 for v in probs.values())
    assert len(a.experts) < 100
    assert np.mean(predicted[100:140]) > np.mean(predicted[25:60])
    assert np.allclose(nonconformity(np.array([0., 2., 5.]),
                                      np.array([-1., -1., 3.]),
                                      np.array([1., 1., 7.])), [0., 2., 0.])
    # Explicit interval nesting and inclusion equivalence.
    for y in [4., 8., 10., 14., 16.]:
        s = abs(y - 10.)
        for r in [0., 2., 4., 6.]:
            assert ((10. - r <= y <= 10. + r) == (s <= r))
    # No score width normalization: same point residual gives same score.
    assert nonconformity(np.array(12.), np.array(9.), np.array(11.)) == 2.
    assert nonconformity(np.array(12.), np.array(0.), np.array(20.)) == 2.
    print("PASS: SAOCP expert updates, probabilities, shifts, absolute scores, nested intervals")


def main() -> None:
    if RUN_SELF_TEST:
        self_test()
        return
    if TRAIN_EVERY < 1:
        raise ValueError("TRAIN_EVERY must be >= 1")
    if SCALER_MODE != "prefix":
        raise ValueError("Only leakage-free SCALER_MODE='prefix' is supported")
    if not DEFAULT_DATASETS:
        raise ValueError("DEFAULT_DATASETS must include at least one file")
    missing = [p for p in DEFAULT_DATASETS if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("Datasets missing (no synthetic fallback):\n" + "\n".join(missing))
    metrics = []
    for path in DEFAULT_DATASETS:
        row = run_dataset(path)
        metrics.append(row)
        pd.DataFrame(metrics).to_csv(OUTPUT_CSV, index=False)
    # Match the column ordering in your ACI/DtACI/AgACI/LACI notebook.
    results_df = pd.DataFrame(metrics)
    metric_order = ["dataset", "method", "PICP", "MPIW", "MLPICP", "LPICE",
                    "LPIA", "LMPIW", "LCWC", "STD_LE", "n_local_windows",
                    "time_sec"]
    ordered = [k for k in metric_order if k in results_df.columns]
    results_df = results_df[ordered + [k for k in results_df if k not in ordered]]
    results_df.to_csv(OUTPUT_CSV, index=False)
    print("\nSaved summary:", OUTPUT_CSV)
    if SAVE_TRAJECTORIES:
        print("Saved trajectories:", TRAJECTORY_DIR)
    print(results_df.to_string(index=False))


if __name__ == "__main__":
    main()



hydropower_dataset: raw=1753 sequences=1705 tinit=200 training=100 initial_cal=100 device=cuda
  hydropower_dataset: 200/1505 | radius=0.76407 | running_coverage=0.9200
  hydropower_dataset: 400/1505 | radius=0.86712 | running_coverage=0.9325
  hydropower_dataset: 600/1505 | radius=0.77366 | running_coverage=0.9400
  hydropower_dataset: 800/1505 | radius=3.30470 | running_coverage=0.9325
  hydropower_dataset: 1000/1505 | radius=1.25469 | running_coverage=0.9360
  hydropower_dataset: 1200/1505 | radius=2.86925 | running_coverage=0.9333
  hydropower_dataset: 1400/1505 | radius=1.03887 | running_coverage=0.9386
[hydropower_dataset/SAOCP] PICP=0.9382 | MPIW=167.6916 | MLPICP=0.9389 | LPICE=0.0069 | LPIA=0.8387 | LMPIW=166.5164 | LCWC=16.3824 | STD_LE=0.0210 | time_sec=6.58

Saved summary: ./results_SAOCP_hydropower_paper.csv
Saved trajectories: ./trajectories/LSTM_SAOCP_hydropower_paper
               dataset method     PICP       MPIW   MLPICP    LPICE     LPIA      LMPIW      LCWC   STD

In [ ]:

from __future__ import annotations

import math
import random
import time
import sys
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd


SEED = 42
SEQ_LEN = 48
EPOCHS = 200
LR = 0.001
BATCH_SIZE = 32
ALPHA = 0.10
SPLIT = 0.50
LIFETIME = 8             # g in paper Eq. (8), authors' official default
TRAIN_EVERY = 6           # preserve notebook update cadence
SAVE_TRAJECTORIES = True
# All settings are configured here. No command-line arguments are required.
TINIT_OVERRIDE = 5000       # same fixed TINIT as the two-location experiment
SCALER_MODE = "prefix"     # fit only on observations available before online evaluation
DEVICE = None             # None selects CUDA automatically when available
LOG_EVERY = 200

# The local window and CWC eta are imported from the SAME helper as ACI/LACI.
# Do not define alternate local-metric formulas here.
TRAJECTORY_DIR = "./trajectories/LSTM_SAOCP_2locations_paper"
OUTPUT_CSV = "./results_SAOCP_2locations_paper.csv"
RUN_SELF_TEST = False      # True runs internal algorithm checks instead of experiments

DEFAULT_DATASETS = [
    "../Datasets/Location1.csv",
    "../Datasets/Location2.csv",
]
LOCATION_NAMES = {"Location1", "Location2"}


def pinball(y: float, s: float, coverage: float) -> float:
    """Paper (1): (1-alpha)-pinball loss with y=true score, s=radius."""
    return max(coverage * (y - s), (1.0 - coverage) * (s - y))


def pinball_grad(y: float, s: float, coverage: float) -> float:
    """Authors' tie convention: zero subgradient when y == s."""
    return (-coverage if y > s else (1.0 - coverage if y < s else 0.0))


def two_adic_valuation(n: int) -> int:
    assert n >= 1
    return (n & -n).bit_length() - 1


@dataclass
class SFOGDExpert:
    """Individual SF-OGD expert; radius is in normalized target units."""
    start: int
    scale: float
    alpha: float
    radius: float
    g: int
    grad_norm: float = 0.0
    z: float = 0.0
    wz: float = 0.0
    age: int = 0

    @property
    def lifetime(self) -> int:
        return self.g * (2 ** two_adic_valuation(self.start))

    @property
    def expired(self) -> bool:
        # Matches author's code: expired when age > lifetime.
        return self.age > self.lifetime

    @property
    def wealth_weight(self) -> float:
        return 0.0 if self.age == 0 else (self.z / self.age) * (1.0 + self.wz)

    def update(self, score: float, meta_loss: float) -> None:
        old_weight = self.wealth_weight
        loss_expert = pinball(score, self.radius, 1.0 - self.alpha)
        # Matches official _OGD.update, including its normalization and clipping.
        gain = (meta_loss - loss_expert) / self.scale / max(self.alpha, 1.0 - self.alpha)
        gain = float(np.clip(gain, -float(old_weight > 0.0), 1.0))
        self.z += gain
        self.wz += gain * old_weight
        self.age += 1
        grad = pinball_grad(score, self.radius, 1.0 - self.alpha)
        self.grad_norm += grad * grad
        if self.grad_norm != 0.0:
            eta = self.scale / math.sqrt(3.0)
            self.radius = max(0.0, self.radius - eta * grad / math.sqrt(self.grad_norm))


class SAOCP:
    """One-step deterministic SAOCP following salesforce/online_conformal/saocp.py.

    Call predict() before seeing the current y. Then call update(score).
    update() creates an expert at the current time, aggregates meta loss,
    updates active experts and increments time; it never sees future scores.
    """
    def __init__(self, alpha: float, scale: float, lifetime: int = LIFETIME):
        if not 0.0 < alpha < 1.0:
            raise ValueError("alpha must lie strictly between zero and one")
        if not np.isfinite(scale) or scale <= 0.0:
            raise ValueError("scale must be finite and positive")
        if lifetime < 1:
            raise ValueError("lifetime must be positive")
        self.alpha = float(alpha)
        self.scale = float(scale)
        self.lifetime = int(lifetime)
        self.t = 1
        self.experts: dict[int, SFOGDExpert] = {}

    def probabilities(self) -> dict[int, float]:
        if not self.experts:
            return {}
        prior = {
            k: 1.0 / (k * k * (1.0 + math.floor(math.log2(k))))
            for k in self.experts
        }
        total_prior = sum(prior.values())
        prior = {k: v / total_prior for k, v in prior.items()}
        unnorm = {k: prior[k] * max(0.0, e.wealth_weight)
                  for k, e in self.experts.items()}
        total = sum(unnorm.values())
        return ({k: v / total for k, v in unnorm.items()}
                if total > 0.0 else prior)

    def predict(self) -> float:
        p = self.probabilities()
        return float(sum(p[k] * self.experts[k].radius for k in p))

    def update(self, score: float) -> None:
        if not np.isfinite(score) or score < 0.0:
            raise ValueError(f"Expected a nonnegative finite conformity score: {score}")
        # Official code initializes the new expert from the CURRENT aggregate,
        # then aggregates once more INCLUDING the newly created expert.
        init_radius = self.predict()
        for k in [k for k, v in self.experts.items() if v.expired]:
            del self.experts[k]
        self.experts[self.t] = SFOGDExpert(
            start=self.t, scale=self.scale, alpha=self.alpha,
            radius=init_radius, g=self.lifetime)
        meta_radius = self.predict()
        meta_loss = pinball(float(score), meta_radius, 1.0 - self.alpha)
        for expert in self.experts.values():
            expert.update(float(score), meta_loss)
        self.t += 1


def point_forecast(lower, upper):
    """Use the midpoint of sorted LSTM quantiles as an H=1 point prediction."""
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)
    return 0.5 * (lower + upper)


def nonconformity(y: np.ndarray, lower: np.ndarray, upper: np.ndarray) -> np.ndarray:
    """Paper Eq. (13), H=1: true radius = absolute point-forecast residual."""
    return np.abs(np.asarray(y, dtype=float) - point_forecast(lower, upper))


def read_series(filepath: str) -> np.ndarray:
    """Match the user's two-location protocol: first 10,000 rows of Power."""
    path = Path(filepath)
    if not path.is_file():
        raise FileNotFoundError(f"Missing dataset: {path}")
    if path.stem not in LOCATION_NAMES:
        raise ValueError(f"Expected Location1 or Location2: {path}")
    df = pd.read_csv(path).iloc[:10000]
    if "Power" not in df.columns:
        raise ValueError(f"{path}: missing 'Power' column")
    y = pd.to_numeric(df["Power"], errors="raise").to_numpy(dtype=float)
    if not len(y) or not np.all(np.isfinite(y)):
        raise ValueError(f"Nonfinite or empty Power series: {path}")
    return y


def build_sequences(series: np.ndarray, seq_len: int) -> tuple[np.ndarray, np.ndarray]:
    # A sliding view avoids the very slow Python list construction in notebook.
    if len(series) <= seq_len:
        raise ValueError("Series too short for SEQ_LEN")
    windows = np.lib.stride_tricks.sliding_window_view(series, seq_len + 1)
    X = np.ascontiguousarray(windows[:, :-1], dtype=np.float32)
    y = np.asarray(windows[:, -1], dtype=np.float32)
    return X, y


def seed_everything(seed: int) -> None:
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def setup_models(device: str):
    import torch
    if str(Path.cwd()) not in sys.path:
        sys.path.insert(0, str(Path.cwd()))
    try:
        from Helper_Scripts.models_TLMR import LSTMQuantileReg
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Missing Helper_Scripts.models_TLMR.LSTMQuantileReg; run alongside "
            "your existing experiment or adjust import in setup_models()."
        ) from exc
    models = [LSTMQuantileReg().to(device) for _ in range(2)]
    opts = [torch.optim.Adam(m.parameters(), lr=LR) for m in models]
    return models, opts


def fit_models(models, opts, X: np.ndarray, y: np.ndarray, epochs: int,
               device: str, alpha: float) -> None:
    import torch
    from torch.utils.data import DataLoader, TensorDataset
    if epochs < 1:
        return
    data = TensorDataset(torch.as_tensor(X, dtype=torch.float32),
                         torch.as_tensor(y.reshape(-1, 1), dtype=torch.float32))
    loader = DataLoader(data, batch_size=min(BATCH_SIZE, len(data)), shuffle=True)
    for model, opt, q in zip(models, opts, [alpha / 2.0, 1.0 - alpha / 2.0]):
        model.train()
        for _ in range(epochs):
            for bx, by in loader:
                bx, by = bx.to(device), by.to(device)
                pred = model(bx)
                diff = by - pred
                loss = torch.maximum(q * diff, (q - 1.0) * diff).mean()
                opt.zero_grad()
                loss.backward()
                opt.step()


def predict_quantiles(models, X: np.ndarray, device: str,
                      batch_size: int = 4096) -> tuple[np.ndarray, np.ndarray]:
    import torch
    outputs = []
    for model in models:
        model.eval()
        chunks = []
        with torch.no_grad():
            for idx in range(0, len(X), batch_size):
                bx = torch.as_tensor(X[idx:idx + batch_size], dtype=torch.float32,
                                     device=device)
                chunks.append(model(bx).detach().cpu().numpy().reshape(-1))
        outputs.append(np.concatenate(chunks).astype(float))
    # Crossing correction to guarantee a valid nested interval family.
    return np.minimum(outputs[0], outputs[1]), np.maximum(outputs[0], outputs[1])


def rolling_local_trajectories(traj: pd.DataFrame, window: int, alpha: float) -> None:
    """Store per-time local diagnostics without changing the metric definitions.

    Only complete, trailing windows are reported (the first window-1 rows
    remain NaN), matching the usual local-coverage evaluation convention.
    Summary metrics are computed exclusively by the original shared helper.
    """
    if window < 1:
        raise ValueError("WINDOW from calculate_metrics must be positive")
    covered = ((traj["y_l_pred"] <= traj["y_true"]) &
               (traj["y_true"] <= traj["y_u_pred"])).astype(float)
    width = traj["y_u_pred"] - traj["y_l_pred"]
    lp = covered.rolling(window=window, min_periods=window).mean()
    traj["LPICP"] = lp
    traj["LE"] = ((1.0 - alpha) - lp).clip(lower=0.0)
    traj["local_overcoverage"] = (lp - (1.0 - alpha)).clip(lower=0.0)
    traj["local_abs_coverage_error"] = (lp - (1.0 - alpha)).abs()
    traj["local_MPIW"] = width.rolling(window=window, min_periods=window).mean()


def run_dataset(filepath: str) -> dict:
    from sklearn.preprocessing import StandardScaler
    from Helper_Scripts.calculate_metrics import (
        summarize_all_metrics, WINDOW as LOCAL_WINDOW, CWC_ETA
    )
    import torch
    name = Path(filepath).stem
    start_time = time.monotonic()
    seed_everything(SEED)
    raw = read_series(filepath)
    tinit = TINIT_OVERRIDE
    if len(raw) <= SEQ_LEN + tinit:
        raise ValueError(f"{name}: {len(raw)} points cannot support tinit={tinit}, seq_len={SEQ_LEN}")

    # Prefix mode prevents leakage from future observations into normalization.
    scaler = StandardScaler()
    scaler.fit(raw[:SEQ_LEN + tinit].reshape(-1, 1))
    series = scaler.transform(raw.reshape(-1, 1)).ravel()
    X, y = build_sequences(series, SEQ_LEN)
    n_train = int(round(SPLIT * tinit))
    if n_train < 2 or n_train >= tinit:
        raise ValueError("Train/calibration split must leave both nonempty")

    device = DEVICE or ("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{name}: raw={len(raw)} sequences={len(y)} tinit={tinit} "
          f"training={n_train} initial_cal={tinit-n_train} device={device}", flush=True)
    models, opts = setup_models(device)
    fit_models(models, opts, X[:n_train], y[:n_train], EPOCHS, device, ALPHA)

    # Initial calibration is temporal and held out from LSTM fitting.
    cal_lo, cal_hi = predict_quantiles(models, X[n_train:tinit], device)
    cal_scores = nonconformity(y[n_train:tinit], cal_lo, cal_hi)
    max_score = float(np.max(cal_scores))
    # Official scale rule on absolute (not halfwidth-normalized) residuals.
    # Zero-only calibration needs a strictly positive numerical fallback.
    scale = max(max_score * math.sqrt(3), 1e-8)
    saocp = SAOCP(alpha=ALPHA, scale=scale, lifetime=LIFETIME)
    for score in cal_scores:
        saocp.update(float(score))

    rows = []
    for t in range(tinit, len(y)):
        # Keep the existing periodic base-model fine-tuning protocol.
        if t > tinit and (t - tinit) % TRAIN_EVERY == 0:
            # Most recent n_train examples with labels revealed before t.
            fit_models(models, opts, X[t-n_train:t], y[t-n_train:t],
                       1, device, ALPHA)

        lo_q, hi_q = predict_quantiles(models, X[t:t+1], device)
        lo_q, hi_q = float(lo_q[0]), float(hi_q[0])
        radius = saocp.predict()  # BEFORE observing y[t], units: normalized target
        center = float(point_forecast(lo_q, hi_q))
        lower = center - radius
        upper = center + radius
        yt = float(y[t])
        err = int(yt < lower or yt > upper)
        score = abs(yt - center)  # paper true radius, unscaled by quantile width
        rows.append((t - tinit, t, yt, lower, upper, center, lo_q, hi_q,
                     radius, score, err, len(saocp.experts)))
        saocp.update(score)       # AFTER observing y[t]
        if LOG_EVERY and (t - tinit + 1) % LOG_EVERY == 0:
            print(f"  {name}: {t-tinit+1}/{len(y)-tinit} | "
                  f"radius={radius:.5f} | running_coverage="
                  f"{1.0-np.mean([r[10] for r in rows]):.4f}", flush=True)

    columns = ["step", "dataset_index", "y_true", "y_l_pred", "y_u_pred",
               "point_pred", "q_lower", "q_upper", "radius", "score", "err", "active_experts"]
    traj = pd.DataFrame(rows, columns=columns)
    # Invert scale only for target-valued quantities, not counts/indicators.
    for col in ["y_true", "y_l_pred", "y_u_pred", "point_pred", "q_lower", "q_upper"]:
        traj[col] = scaler.inverse_transform(traj[[col]].to_numpy()).reshape(-1)
    traj["radius"] *= float(scaler.scale_[0])
    traj["score"] *= float(scaler.scale_[0])
    traj["alphaTrajectory"] = np.nan  # SAOCP directly learns radius, not alpha_t.
    traj["errSeq"] = traj["err"]
    traj["covered"] = 1 - traj["err"]
    rolling_local_trajectories(traj, LOCAL_WINDOW, ALPHA)

    outdir = Path(TRAJECTORY_DIR)
    outdir.mkdir(parents=True, exist_ok=True)
    if SAVE_TRAJECTORIES:
        traj.to_csv(outdir / f"{name}__SAOCP.csv", index=False)

    truth = traj["y_true"].to_numpy(dtype=float)
    lower = traj["y_l_pred"].to_numpy(dtype=float)
    upper = traj["y_u_pred"].to_numpy(dtype=float)
    elapsed = time.monotonic() - start_time
    # Same evaluator, arguments and metric names as the ACI/LACI experiment.
    metrics = summarize_all_metrics(
        truth, lower, upper,
        alpha=ALPHA, w=LOCAL_WINDOW, eta=CWC_ETA, time_taken=elapsed
    )
    metrics["time_sec"] = float(elapsed)
    metrics["dataset"] = name + ".csv"
    metrics["method"] = "SAOCP"
    metrics["scale_D_from_cal"] = float(max_score * scaler.scale_[0])  # original units
    metrics["g_lifetime"] = LIFETIME
    metrics["mean_radius"] = float(traj["radius"].mean())
    metrics["median_radius"] = float(traj["radius"].median())
    base_covered = ((traj["y_true"] >= traj["q_lower"]) &
                    (traj["y_true"] <= traj["q_upper"]))
    metrics["base_quantile_PICP"] = float(base_covered.mean())
    metric_names = ["PICP", "MPIW", "MLPICP", "LPICE", "LPIA",
                    "LMPIW", "LCWC", "STD_LE"]
    print(f"[{name}/SAOCP] " + " | ".join(
        f"{k}={float(metrics[k]):.4f}" for k in metric_names if k in metrics
    ) + f" | time_sec={elapsed:.2f}", flush=True)
    return metrics


def self_test() -> None:
    # Regression tests of the deterministic official-style SAOCP machinery.
    a = SAOCP(alpha=0.1, scale=math.sqrt(3), lifetime=8)
    scores = np.r_[np.full(70, 0.5), np.full(70, 2.0), np.full(70, 0.3)]
    predicted = []
    for score in scores:
        r = a.predict()
        predicted.append(r)
        assert np.isfinite(r) and r >= 0
        a.update(float(score))
        probs = a.probabilities()
        assert abs(sum(probs.values()) - 1.0) < 1e-9
        assert all(v >= 0.0 for v in probs.values())
    assert len(a.experts) < 100
    assert np.mean(predicted[100:140]) > np.mean(predicted[25:60])
    assert np.allclose(nonconformity(np.array([0., 2., 5.]),
                                      np.array([-1., -1., 3.]),
                                      np.array([1., 1., 7.])), [0., 2., 0.])
    # Explicit interval nesting and inclusion equivalence.
    for y in [4., 8., 10., 14., 16.]:
        s = abs(y - 10.)
        for r in [0., 2., 4., 6.]:
            assert ((10. - r <= y <= 10. + r) == (s <= r))
    # No score width normalization: same point residual gives same score.
    assert nonconformity(np.array(12.), np.array(9.), np.array(11.)) == 2.
    assert nonconformity(np.array(12.), np.array(0.), np.array(20.)) == 2.
    print("PASS: SAOCP expert updates, probabilities, shifts, absolute scores, nested intervals")


def main() -> None:
    if RUN_SELF_TEST:
        self_test()
        return
    if TRAIN_EVERY < 1:
        raise ValueError("TRAIN_EVERY must be >= 1")
    if SCALER_MODE != "prefix":
        raise ValueError("Only leakage-free SCALER_MODE='prefix' is supported")
    if not DEFAULT_DATASETS:
        raise ValueError("DEFAULT_DATASETS must include at least one file")
    missing = [p for p in DEFAULT_DATASETS if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("Datasets missing (no synthetic fallback):\n" + "\n".join(missing))
    metrics = []
    for path in DEFAULT_DATASETS:
        row = run_dataset(path)
        metrics.append(row)
        pd.DataFrame(metrics).to_csv(OUTPUT_CSV, index=False)
    # Match the column ordering in your ACI/DtACI/AgACI/LACI notebook.
    results_df = pd.DataFrame(metrics)
    metric_order = ["dataset", "method", "PICP", "MPIW", "MLPICP", "LPICE",
                    "LPIA", "LMPIW", "LCWC", "STD_LE", "n_local_windows",
                    "time_sec"]
    ordered = [k for k in metric_order if k in results_df.columns]
    results_df = results_df[ordered + [k for k in results_df if k not in ordered]]
    results_df.to_csv(OUTPUT_CSV, index=False)
    print("\nSaved summary:", OUTPUT_CSV)
    if SAVE_TRAJECTORIES:
        print("Saved trajectories:", TRAJECTORY_DIR)
    print(results_df.to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:


from __future__ import annotations

import math
import random
import time
import sys
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd


SEED = 42
SEQ_LEN = 48
EPOCHS = 200
LR = 0.001
BATCH_SIZE = 32
ALPHA = 0.10
SPLIT = 0.50
LIFETIME = 8             # g in paper Eq. (8), authors' official default
TRAIN_EVERY = 6           # preserve notebook update cadence
SAVE_TRAJECTORIES = True
# All settings are configured here. No command-line arguments are required.
TINIT_OVERRIDE = 4000       # same fixed TINIT as the solar experiment
SCALER_MODE = "prefix"     # fit only on observations available before online evaluation
DEVICE = None             # None selects CUDA automatically when available
LOG_EVERY = 200

# The local window and CWC eta are imported from the SAME helper as ACI/LACI.
# Do not define alternate local-metric formulas here.
TRAJECTORY_DIR = "./trajectories/LSTM_SAOCP_2plants_paper"
OUTPUT_CSV = "./results_SAOCP_2plants_paper.csv"
RUN_SELF_TEST = False      # True runs internal algorithm checks instead of experiments

DEFAULT_DATASETS = [
    "../Datasets/Plant_1_Generation_Data.csv",
    "../Datasets/Plant_2_Generation_Data.csv",
]
SOLAR_NAMES = {"Plant_1_Generation_Data", "Plant_2_Generation_Data"}


def pinball(y: float, s: float, coverage: float) -> float:
    """Paper (1): (1-alpha)-pinball loss with y=true score, s=radius."""
    return max(coverage * (y - s), (1.0 - coverage) * (s - y))


def pinball_grad(y: float, s: float, coverage: float) -> float:
    """Authors' tie convention: zero subgradient when y == s."""
    return (-coverage if y > s else (1.0 - coverage if y < s else 0.0))


def two_adic_valuation(n: int) -> int:
    assert n >= 1
    return (n & -n).bit_length() - 1


@dataclass
class SFOGDExpert:
    """Individual SF-OGD expert; radius is in normalized target units."""
    start: int
    scale: float
    alpha: float
    radius: float
    g: int
    grad_norm: float = 0.0
    z: float = 0.0
    wz: float = 0.0
    age: int = 0

    @property
    def lifetime(self) -> int:
        return self.g * (2 ** two_adic_valuation(self.start))

    @property
    def expired(self) -> bool:
        # Matches author's code: expired when age > lifetime.
        return self.age > self.lifetime

    @property
    def wealth_weight(self) -> float:
        return 0.0 if self.age == 0 else (self.z / self.age) * (1.0 + self.wz)

    def update(self, score: float, meta_loss: float) -> None:
        old_weight = self.wealth_weight
        loss_expert = pinball(score, self.radius, 1.0 - self.alpha)
        # Matches official _OGD.update, including its normalization and clipping.
        gain = (meta_loss - loss_expert) / self.scale / max(self.alpha, 1.0 - self.alpha)
        gain = float(np.clip(gain, -float(old_weight > 0.0), 1.0))
        self.z += gain
        self.wz += gain * old_weight
        self.age += 1
        grad = pinball_grad(score, self.radius, 1.0 - self.alpha)
        self.grad_norm += grad * grad
        if self.grad_norm != 0.0:
            eta = self.scale / math.sqrt(3.0)
            self.radius = max(0.0, self.radius - eta * grad / math.sqrt(self.grad_norm))


class SAOCP:
    """One-step deterministic SAOCP following salesforce/online_conformal/saocp.py.

    Call predict() before seeing the current y. Then call update(score).
    update() creates an expert at the current time, aggregates meta loss,
    updates active experts and increments time; it never sees future scores.
    """
    def __init__(self, alpha: float, scale: float, lifetime: int = LIFETIME):
        if not 0.0 < alpha < 1.0:
            raise ValueError("alpha must lie strictly between zero and one")
        if not np.isfinite(scale) or scale <= 0.0:
            raise ValueError("scale must be finite and positive")
        if lifetime < 1:
            raise ValueError("lifetime must be positive")
        self.alpha = float(alpha)
        self.scale = float(scale)
        self.lifetime = int(lifetime)
        self.t = 1
        self.experts: dict[int, SFOGDExpert] = {}

    def probabilities(self) -> dict[int, float]:
        if not self.experts:
            return {}
        prior = {
            k: 1.0 / (k * k * (1.0 + math.floor(math.log2(k))))
            for k in self.experts
        }
        total_prior = sum(prior.values())
        prior = {k: v / total_prior for k, v in prior.items()}
        unnorm = {k: prior[k] * max(0.0, e.wealth_weight)
                  for k, e in self.experts.items()}
        total = sum(unnorm.values())
        return ({k: v / total for k, v in unnorm.items()}
                if total > 0.0 else prior)

    def predict(self) -> float:
        p = self.probabilities()
        return float(sum(p[k] * self.experts[k].radius for k in p))

    def update(self, score: float) -> None:
        if not np.isfinite(score) or score < 0.0:
            raise ValueError(f"Expected a nonnegative finite conformity score: {score}")
        # Official code initializes the new expert from the CURRENT aggregate,
        # then aggregates once more INCLUDING the newly created expert.
        init_radius = self.predict()
        for k in [k for k, v in self.experts.items() if v.expired]:
            del self.experts[k]
        self.experts[self.t] = SFOGDExpert(
            start=self.t, scale=self.scale, alpha=self.alpha,
            radius=init_radius, g=self.lifetime)
        meta_radius = self.predict()
        meta_loss = pinball(float(score), meta_radius, 1.0 - self.alpha)
        for expert in self.experts.values():
            expert.update(float(score), meta_loss)
        self.t += 1


def point_forecast(lower, upper):
    """Use the midpoint of sorted LSTM quantiles as an H=1 point prediction."""
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)
    return 0.5 * (lower + upper)


def nonconformity(y: np.ndarray, lower: np.ndarray, upper: np.ndarray) -> np.ndarray:
    """Paper Eq. (13), H=1: true radius = absolute point-forecast residual."""
    return np.abs(np.asarray(y, dtype=float) - point_forecast(lower, upper))


def read_series(filepath: str) -> np.ndarray:
    """Replicate solar data setup: first 10000 rows, nonzero DAILY_YIELD.

    Filtering removes observations; sequential predictions refer to successive
    *retained* rows, not necessarily consecutive timestamps.
    """
    path = Path(filepath)
    if not path.is_file():
        raise FileNotFoundError(f"Missing dataset: {path}")
    if path.stem not in SOLAR_NAMES:
        raise ValueError(f"Expected Plant 1 or Plant 2 file: {path}")
    df = pd.read_csv(path).iloc[:10000]
    if "DAILY_YIELD" not in df.columns:
        raise ValueError(f"{path}: missing 'DAILY_YIELD' column")
    values = pd.to_numeric(df["DAILY_YIELD"], errors="coerce")
    y = values.loc[values.notna() & (values != 0)].to_numpy(dtype=float)
    if not len(y) or not np.all(np.isfinite(y)):
        raise ValueError(f"Nonfinite or empty filtered DAILY_YIELD series: {path}")
    return y


def build_sequences(series: np.ndarray, seq_len: int) -> tuple[np.ndarray, np.ndarray]:
    # A sliding view avoids the very slow Python list construction in notebook.
    if len(series) <= seq_len:
        raise ValueError("Series too short for SEQ_LEN")
    windows = np.lib.stride_tricks.sliding_window_view(series, seq_len + 1)
    X = np.ascontiguousarray(windows[:, :-1], dtype=np.float32)
    y = np.asarray(windows[:, -1], dtype=np.float32)
    return X, y


def seed_everything(seed: int) -> None:
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def setup_models(device: str):
    import torch
    if str(Path.cwd()) not in sys.path:
        sys.path.insert(0, str(Path.cwd()))
    try:
        from Helper_Scripts.models_TLMR import LSTMQuantileReg
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Missing Helper_Scripts.models_TLMR.LSTMQuantileReg; run alongside "
            "your existing experiment or adjust import in setup_models()."
        ) from exc
    models = [LSTMQuantileReg().to(device) for _ in range(2)]
    opts = [torch.optim.Adam(m.parameters(), lr=LR) for m in models]
    return models, opts


def fit_models(models, opts, X: np.ndarray, y: np.ndarray, epochs: int,
               device: str, alpha: float) -> None:
    import torch
    from torch.utils.data import DataLoader, TensorDataset
    if epochs < 1:
        return
    data = TensorDataset(torch.as_tensor(X, dtype=torch.float32),
                         torch.as_tensor(y.reshape(-1, 1), dtype=torch.float32))
    loader = DataLoader(data, batch_size=min(BATCH_SIZE, len(data)), shuffle=True)
    for model, opt, q in zip(models, opts, [alpha / 2.0, 1.0 - alpha / 2.0]):
        model.train()
        for _ in range(epochs):
            for bx, by in loader:
                bx, by = bx.to(device), by.to(device)
                pred = model(bx)
                diff = by - pred
                loss = torch.maximum(q * diff, (q - 1.0) * diff).mean()
                opt.zero_grad()
                loss.backward()
                opt.step()


def predict_quantiles(models, X: np.ndarray, device: str,
                      batch_size: int = 4096) -> tuple[np.ndarray, np.ndarray]:
    import torch
    outputs = []
    for model in models:
        model.eval()
        chunks = []
        with torch.no_grad():
            for idx in range(0, len(X), batch_size):
                bx = torch.as_tensor(X[idx:idx + batch_size], dtype=torch.float32,
                                     device=device)
                chunks.append(model(bx).detach().cpu().numpy().reshape(-1))
        outputs.append(np.concatenate(chunks).astype(float))
    # Crossing correction to guarantee a valid nested interval family.
    return np.minimum(outputs[0], outputs[1]), np.maximum(outputs[0], outputs[1])


def rolling_local_trajectories(traj: pd.DataFrame, window: int, alpha: float) -> None:
    """Store per-time local diagnostics without changing the metric definitions.

    Only complete, trailing windows are reported (the first window-1 rows
    remain NaN), matching the usual local-coverage evaluation convention.
    Summary metrics are computed exclusively by the original shared helper.
    """
    if window < 1:
        raise ValueError("WINDOW from calculate_metrics must be positive")
    covered = ((traj["y_l_pred"] <= traj["y_true"]) &
               (traj["y_true"] <= traj["y_u_pred"])).astype(float)
    width = traj["y_u_pred"] - traj["y_l_pred"]
    lp = covered.rolling(window=window, min_periods=window).mean()
    traj["LPICP"] = lp
    traj["LE"] = ((1.0 - alpha) - lp).clip(lower=0.0)
    traj["local_overcoverage"] = (lp - (1.0 - alpha)).clip(lower=0.0)
    traj["local_abs_coverage_error"] = (lp - (1.0 - alpha)).abs()
    traj["local_MPIW"] = width.rolling(window=window, min_periods=window).mean()


def run_dataset(filepath: str) -> dict:
    from sklearn.preprocessing import StandardScaler
    from Helper_Scripts.calculate_metrics import (
        summarize_all_metrics, WINDOW as LOCAL_WINDOW, CWC_ETA
    )
    import torch
    name = Path(filepath).stem
    start_time = time.monotonic()
    seed_everything(SEED)
    raw = read_series(filepath)
    tinit = TINIT_OVERRIDE
    if len(raw) <= SEQ_LEN + tinit:
        raise ValueError(f"{name}: {len(raw)} points cannot support tinit={tinit}, seq_len={SEQ_LEN}")

    # Prefix mode prevents leakage from future observations into normalization.
    scaler = StandardScaler()
    scaler.fit(raw[:SEQ_LEN + tinit].reshape(-1, 1))
    series = scaler.transform(raw.reshape(-1, 1)).ravel()
    X, y = build_sequences(series, SEQ_LEN)
    n_train = int(round(SPLIT * tinit))
    if n_train < 2 or n_train >= tinit:
        raise ValueError("Train/calibration split must leave both nonempty")

    device = DEVICE or ("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{name}: raw={len(raw)} sequences={len(y)} tinit={tinit} "
          f"training={n_train} initial_cal={tinit-n_train} device={device}", flush=True)
    models, opts = setup_models(device)
    fit_models(models, opts, X[:n_train], y[:n_train], EPOCHS, device, ALPHA)

    # Initial calibration is temporal and held out from LSTM fitting.
    cal_lo, cal_hi = predict_quantiles(models, X[n_train:tinit], device)
    cal_scores = nonconformity(y[n_train:tinit], cal_lo, cal_hi)
    max_score = float(np.max(cal_scores))
    # Official scale rule on absolute (not halfwidth-normalized) residuals.
    # Zero-only calibration needs a strictly positive numerical fallback.
    scale = max(max_score * math.sqrt(3), 1e-8)
    saocp = SAOCP(alpha=ALPHA, scale=scale, lifetime=LIFETIME)
    for score in cal_scores:
        saocp.update(float(score))

    rows = []
    for t in range(tinit, len(y)):
        # Keep the existing periodic base-model fine-tuning protocol.
        if t > tinit and (t - tinit) % TRAIN_EVERY == 0:
            # Most recent n_train examples with labels revealed before t.
            fit_models(models, opts, X[t-n_train:t], y[t-n_train:t],
                       1, device, ALPHA)

        lo_q, hi_q = predict_quantiles(models, X[t:t+1], device)
        lo_q, hi_q = float(lo_q[0]), float(hi_q[0])
        radius = saocp.predict()  # BEFORE observing y[t], units: normalized target
        center = float(point_forecast(lo_q, hi_q))
        lower = center - radius
        upper = center + radius
        yt = float(y[t])
        err = int(yt < lower or yt > upper)
        score = abs(yt - center)  # paper true radius, unscaled by quantile width
        rows.append((t - tinit, t, yt, lower, upper, center, lo_q, hi_q,
                     radius, score, err, len(saocp.experts)))
        saocp.update(score)       # AFTER observing y[t]
        if LOG_EVERY and (t - tinit + 1) % LOG_EVERY == 0:
            print(f"  {name}: {t-tinit+1}/{len(y)-tinit} | "
                  f"radius={radius:.5f} | running_coverage="
                  f"{1.0-np.mean([r[10] for r in rows]):.4f}", flush=True)

    columns = ["step", "dataset_index", "y_true", "y_l_pred", "y_u_pred",
               "point_pred", "q_lower", "q_upper", "radius", "score", "err", "active_experts"]
    traj = pd.DataFrame(rows, columns=columns)
    # Invert scale only for target-valued quantities, not counts/indicators.
    for col in ["y_true", "y_l_pred", "y_u_pred", "point_pred", "q_lower", "q_upper"]:
        traj[col] = scaler.inverse_transform(traj[[col]].to_numpy()).reshape(-1)
    traj["radius"] *= float(scaler.scale_[0])
    traj["score"] *= float(scaler.scale_[0])
    traj["alphaTrajectory"] = np.nan  # SAOCP directly learns radius, not alpha_t.
    traj["errSeq"] = traj["err"]
    traj["covered"] = 1 - traj["err"]
    rolling_local_trajectories(traj, LOCAL_WINDOW, ALPHA)

    outdir = Path(TRAJECTORY_DIR)
    outdir.mkdir(parents=True, exist_ok=True)
    if SAVE_TRAJECTORIES:
        traj.to_csv(outdir / f"{name}__SAOCP.csv", index=False)

    truth = traj["y_true"].to_numpy(dtype=float)
    lower = traj["y_l_pred"].to_numpy(dtype=float)
    upper = traj["y_u_pred"].to_numpy(dtype=float)
    elapsed = time.monotonic() - start_time
    # Same evaluator, arguments and metric names as the ACI/LACI experiment.
    metrics = summarize_all_metrics(
        truth, lower, upper,
        alpha=ALPHA, w=LOCAL_WINDOW, eta=CWC_ETA, time_taken=elapsed
    )
    metrics["time_sec"] = float(elapsed)
    metrics["dataset"] = name + ".csv"
    metrics["method"] = "SAOCP"
    metrics["scale_D_from_cal"] = float(max_score * scaler.scale_[0])  # original units
    metrics["g_lifetime"] = LIFETIME
    metrics["mean_radius"] = float(traj["radius"].mean())
    metrics["median_radius"] = float(traj["radius"].median())
    base_covered = ((traj["y_true"] >= traj["q_lower"]) &
                    (traj["y_true"] <= traj["q_upper"]))
    metrics["base_quantile_PICP"] = float(base_covered.mean())
    metric_names = ["PICP", "MPIW", "MLPICP", "LPICE", "LPIA",
                    "LMPIW", "LCWC", "STD_LE"]
    print(f"[{name}/SAOCP] " + " | ".join(
        f"{k}={float(metrics[k]):.4f}" for k in metric_names if k in metrics
    ) + f" | time_sec={elapsed:.2f}", flush=True)
    return metrics


def self_test() -> None:
    # Regression tests of the deterministic official-style SAOCP machinery.
    a = SAOCP(alpha=0.1, scale=math.sqrt(3), lifetime=8)
    scores = np.r_[np.full(70, 0.5), np.full(70, 2.0), np.full(70, 0.3)]
    predicted = []
    for score in scores:
        r = a.predict()
        predicted.append(r)
        assert np.isfinite(r) and r >= 0
        a.update(float(score))
        probs = a.probabilities()
        assert abs(sum(probs.values()) - 1.0) < 1e-9
        assert all(v >= 0.0 for v in probs.values())
    assert len(a.experts) < 100
    assert np.mean(predicted[100:140]) > np.mean(predicted[25:60])
    assert np.allclose(nonconformity(np.array([0., 2., 5.]),
                                      np.array([-1., -1., 3.]),
                                      np.array([1., 1., 7.])), [0., 2., 0.])
    # Explicit interval nesting and inclusion equivalence.
    for y in [4., 8., 10., 14., 16.]:
        s = abs(y - 10.)
        for r in [0., 2., 4., 6.]:
            assert ((10. - r <= y <= 10. + r) == (s <= r))
    # No score width normalization: same point residual gives same score.
    assert nonconformity(np.array(12.), np.array(9.), np.array(11.)) == 2.
    assert nonconformity(np.array(12.), np.array(0.), np.array(20.)) == 2.
    print("PASS: SAOCP expert updates, probabilities, shifts, absolute scores, nested intervals")


def main() -> None:
    if RUN_SELF_TEST:
        self_test()
        return
    if TRAIN_EVERY < 1:
        raise ValueError("TRAIN_EVERY must be >= 1")
    if SCALER_MODE != "prefix":
        raise ValueError("Only leakage-free SCALER_MODE='prefix' is supported")
    if not DEFAULT_DATASETS:
        raise ValueError("DEFAULT_DATASETS must include at least one file")
    missing = [p for p in DEFAULT_DATASETS if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("Datasets missing (no synthetic fallback):\n" + "\n".join(missing))
    metrics = []
    for path in DEFAULT_DATASETS:
        row = run_dataset(path)
        metrics.append(row)
        pd.DataFrame(metrics).to_csv(OUTPUT_CSV, index=False)
    # Match the column ordering in your ACI/DtACI/AgACI/LACI notebook.
    results_df = pd.DataFrame(metrics)
    metric_order = ["dataset", "method", "PICP", "MPIW", "MLPICP", "LPICE",
                    "LPIA", "LMPIW", "LCWC", "STD_LE", "n_local_windows",
                    "time_sec"]
    ordered = [k for k in metric_order if k in results_df.columns]
    results_df = results_df[ordered + [k for k in results_df if k not in ordered]]
    results_df.to_csv(OUTPUT_CSV, index=False)
    print("\nSaved summary:", OUTPUT_CSV)
    if SAVE_TRAJECTORIES:
        print("Saved trajectories:", TRAJECTORY_DIR)
    print(results_df.to_string(index=False))


if __name__ == "__main__":
    main()
